In [1]:
import os
import warnings
import lightkurve as lk
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

safe_dir = os.path.join(
    os.path.expanduser("~"), "Documents", "Exoplanet_Project"
)
os.makedirs(safe_dir, exist_ok=True)
csv_path = os.path.join(safe_dir, "kepler_transit_features.csv")

# ۱. بارگذاری دیتاست ذخیره‌شده
if "dataset_df" in globals() and len(dataset_df) > 0:
  df_work = dataset_df.copy()
else:
  df_work = pd.read_csv(csv_path)

feature_cols = [
    "baseline_flux",
    "min_flux",
    "transit_depth",
    "flux_std",
    "snr_proxy",
    "asymmetry",
]

print("تعداد ردیف‌های دیتاست برای ارزیابی:", len(df_work))
print("تعداد ستاره‌های یکتا (kepid):", df_work["kepid"].nunique())

# ۲. ارزیابی گروهی با GroupKFold (جلوگیری از نشت داده بین ستاره‌ها)
n_splits = min(3, df_work["kepid"].nunique())
results_summary = []
best_rf_model = None

if len(df_work) >= 4 and n_splits >= 2:
  X = df_work[feature_cols].fillna(0.0)
  y = df_work["label"].astype(int)
  groups = df_work["kepid"]

  gkf = GroupKFold(n_splits=n_splits)

  model_factories = {
      "Logistic Regression": lambda: make_pipeline(
          StandardScaler(),
          LogisticRegression(max_iter=1000, random_state=42),
      ),
      "Random Forest": lambda: RandomForestClassifier(
          n_estimators=200, random_state=42
      ),
  }

  for model_name, factory in model_factories.items():
    fold_f1 = []
    fold_acc = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):
      clf = factory()
      clf.fit(X.iloc[train_idx], y.iloc[train_idx])
      preds = clf.predict(X.iloc[val_idx])

      fold_f1.append(f1_score(y.iloc[val_idx], preds, zero_division=0))
      fold_acc.append(accuracy_score(y.iloc[val_idx], preds))

      if model_name == "Random Forest":
        best_rf_model = clf

    results_summary.append({
        "Model": model_name,
        "CV_Folds": n_splits,
        "Mean_F1": round(float(np.mean(fold_f1)), 4),
        "Mean_Accuracy": round(float(np.mean(fold_acc)), 4),
    })

  # آموزش نهایی Random Forest روی کل دیتاست برای پیش‌بینی هدف جدید
  final_rf = RandomForestClassifier(n_estimators=200, random_state=42)
  final_rf.fit(X, y)
  best_rf_model = final_rf

  cv_df = pd.DataFrame(results_summary)
  print("\n=== جدول مقایسه مدل‌ها با GroupKFold ===")
  display(cv_df)
else:
  print("تعداد ستاره‌های یکتا برای GroupKFold کافی نیست.")

# ۳. اطمینان از وجود توابع کمکی پیش‌پردازش برای استنتاج روی نمونه جدید
if "fold_and_extract_window" not in globals():

  def fold_and_extract_window(
      lc_flat, period, epoch_time, phase_half_width=0.05, n_bins=100
  ):
    folded = lc_flat.fold(period=period, epoch_time=epoch_time)
    phase = np.asarray(folded.phase.value, dtype=float)
    flux = np.asarray(folded.flux.value, dtype=float)

    mask = np.abs(phase) <= phase_half_width
    phase_w = phase[mask]
    flux_w = flux[mask]

    if len(flux_w) < 10 or not np.isfinite(flux_w).any():
      return None, None

    bin_edges = np.linspace(-phase_half_width, phase_half_width, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    binned_flux = np.full(n_bins, np.nan, dtype=float)

    for i in range(n_bins):
      in_bin = (phase_w >= bin_edges[i]) & (phase_w < bin_edges[i + 1])
      if np.any(in_bin):
        binned_flux[i] = np.nanmean(flux_w[in_bin])

    valid = np.isfinite(binned_flux)
    if np.sum(valid) < 5:
      return None, None

    binned_flux = np.interp(bin_centers, bin_centers[valid], binned_flux[valid])
    return bin_centers, binned_flux


if "extract_transit_features" not in globals():

  def extract_transit_features(bin_centers, binned_flux):
    baseline_flux = float(np.median(binned_flux))
    min_flux = float(np.min(binned_flux))
    transit_depth = float(max(0.0, baseline_flux - min_flux))
    flux_std = float(np.std(binned_flux))
    snr_proxy = float(transit_depth / flux_std) if flux_std > 0 else 0.0

    left_half = binned_flux[bin_centers < 0]
    right_half = binned_flux[bin_centers > 0]
    min_len = min(len(left_half), len(right_half))
    asymmetry = (
        float(np.mean(np.abs(left_half[:min_len] - right_half[::-1][:min_len])))
        if min_len > 0
        else 0.0
    )

    return {
        "baseline_flux": baseline_flux,
        "min_flux": min_flux,
        "transit_depth": transit_depth,
        "flux_std": flux_std,
        "snr_proxy": snr_proxy,
        "asymmetry": asymmetry,
    }


def predict_new_target(kepid, period, epoch, model):
  target_str = f"KIC {kepid}"
  s = lk.search_lightcurve(target_str, mission="Kepler")
  if len(s) == 0:
    return {"error": f"منحنی نور یافت نشد: {target_str}"}

  lc = s[0].download().remove_nans().remove_outliers(sigma=5).normalize()
  lc_flat = lc.flatten(window_length=401)
  bx, by = fold_and_extract_window(lc_flat, period=period, epoch_time=epoch)
  if by is None:
    return {"error": f"پنجره گذر نامعتبر بود: {target_str}"}

  feats = extract_transit_features(bx, by)
  x_new = pd.DataFrame([[feats[c] for c in feature_cols]], columns=feature_cols)
  prob_1 = float(model.predict_proba(x_new)[0, 1])
  pred_label = int(prob_1 >= 0.5)
  return {
      "kepid": kepid,
      "pred_label": pred_label,
      "prob_confirmed": round(prob_1, 4),
      **feats,
  }


inference_result = None
if best_rf_model is not None:
  try:
    inference_result = predict_new_target(
        kepid=11904151, period=0.837495, epoch=132.040, model=best_rf_model
    )
    print("\n=== نتیجه پیش‌بینی روی نمونه آزمون ===")
    print(inference_result)
  except Exception as e:
    print("\nخطا در پیش‌بینی نمونه آزمون:", e)

# ۴. ذخیره گزارش خلاصه متنی در پوشه امن
report_path = os.path.join(safe_dir, "summary_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
  f.write("Exoplanet Lightcurve Transit - Evaluation & Inference Report\n")
  f.write("=" * 60 + "\n")
  f.write(f"Dataset rows: {len(df_work)}\n")
  f.write(f"Unique kepid groups: {df_work['kepid'].nunique()}\n\n")
  if results_summary:
    f.write("GroupKFold Model Comparison:\n")
    for item in results_summary:
      f.write(f" - {item}\n")
  f.write("\nSample Inference Result:\n")
  f.write(f" {inference_result}\n")

print("\nفایل گزارش خلاصه ذخیره شد در:", report_path)

c:\Users\island\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


تعداد ردیف‌های دیتاست برای ارزیابی: 5
تعداد ستاره‌های یکتا (kepid): 5

=== جدول مقایسه مدل‌ها با GroupKFold ===


,Model,CV_Folds,Mean_F1,Mean_Accuracy
0,Logistic Regression,3,0.5556,0.5000
1,Random Forest,3,0.7778,0.6667



=== نتیجه پیش‌بینی روی نمونه آزمون ===
{'kepid': 11904151, 'pred_label': 1, 'prob_confirmed': 0.87, 'baseline_flux': 1.0000017650102293, 'min_flux': 0.9999278122878688, 'transit_depth': 7.39527223605485e-05, 'flux_std': 2.7934950964439767e-05, 'snr_proxy': 2.647318853528248, 'asymmetry': 3.284632259181519e-05}

فایل گزارش خلاصه ذخیره شد در: C:\Users\island\Documents\Exoplanet_Project\summary_report.txt
